In [ ]:
# !pip install sacrebleu 

In [ ]:
#!/usr/bin/env python3

# Model ensemble scheme under limited time and gpu constraints.

import gc
import json
import logging
import math
import os
import random
import re
import warnings
from contextlib import nullcontext
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional

# Suppress non-critical warnings in Kaggle logs.
warnings.filterwarnings("ignore")

# Keep CPU preprocessing/tokenization from oversubscribing cores.
os.environ.setdefault("OMP_NUM_THREADS", "4")

# Keep Pandas/NumPy CPU-side work predictable.
os.environ.setdefault("MKL_NUM_THREADS", "4")

# Let Hugging Face tokenizers parallelize batch tokenization.
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

import numpy as np
import pandas as pd
import sacrebleu
import torch
from torch.utils.data import DataLoader, Dataset, Sampler
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer


# =========================================================
# 1. Mixed Precision Helper
# =========================================================
def _supports_cuda_bf16(device: Optional[torch.device] = None) -> bool:
    if not torch.cuda.is_available():
        return False
    try:
        if device is not None and device.type == "cuda" and device.index is not None:
            with torch.cuda.device(device.index):
                return bool(getattr(torch.cuda, "is_bf16_supported", lambda: False)())
        return bool(getattr(torch.cuda, "is_bf16_supported", lambda: False)())
    except Exception:
        return False


def _bfloat16_autocast_context(device: torch.device, enabled: bool):
    if enabled and device.type == "cuda" and _supports_cuda_bf16(device):
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
    return nullcontext()


# =========================================================
# 2. Config
# =========================================================
@dataclass
class EnsembleInferenceConfig:
    # -------------------------
    # Data paths
    # -------------------------
    test_data_path: str = "/kaggle/input/deep-past-initiative-machine-translation/test.csv"
    output_dir: str = "/kaggle/working/"

    # -------------------------
    # Model paths
    # -------------------------
    model_a_path: str = "/kaggle/input/datasets/assiaben/final-byt5/byt5-akkadian-optimized-34x"
    model_b_path: str = "/kaggle/input/models/mattiaangeli/byt5-akkadian-mbr-v2/pytorch/default/1"
    model_c_path: str = "/kaggle/input/datasets/dongdaorui/02-full_aug_baseline"
    


    # Shared input truncation settings.
    max_input_length: int = 512

    # Keep batches small because each sample expands into many candidates.
    batch_size: int = 2

    # Worker processes used by the DataLoader for batch preparation.
    num_workers: int = 2

    # Length buckets reduce padding waste.
    num_buckets: int = 6

    # Per-model generation length caps.
    # Maximum generated tokens per sample for Model-A.
    model_a_max_new_tokens: int = 384
    # Maximum generated tokens per sample for Model-B.
    model_b_max_new_tokens: int = 384
    # Maximum generated tokens per sample for Model-C.
    model_c_max_new_tokens: int = 384

    # Beam search candidates.
    # Number of beam-search sequences kept as candidates for MBR.
    num_beam_cands: int = 4

    # Internal beam width; must be at least num_beam_cands.
    num_beams: int = 8

    # Values above 1.0 discourage overly short outputs.
    length_penalty: float = 1.3

    # Stop beam search when all active beams have finished.
    early_stopping: bool = True

    # Discourages repeated tokens/phrases during generation.
    repetition_penalty: float = 1.2

    # Optional diverse beam candidates.
    # Whether to run grouped diverse beam search in addition to normal beams.
    use_diverse_beam: bool = False

    # Number of diverse-beam candidates to add per sample when enabled.
    num_diverse_cands: int = 4

    # Total beam width for diverse beam search.
    num_diverse_beams: int = 8

    # num_diverse_beams must be divisible by this value.
    num_beam_groups: int = 4

    # Penalty that pushes beam groups away from choosing the same tokens.
    diversity_penalty: float = 0.8

    # Sampling candidates add diversity before MBR.
    # Whether to add nucleus-sampling candidates in addition to beam outputs.
    use_sampling: bool = True

    # Lower temperatures are safer; higher temperatures add variety.
    sample_temperatures: List[float] = field(default_factory=lambda: [0.60, 0.80, 1.05])

    # Number of sampled candidates generated at each temperature.
    num_sample_per_temp: int = 2

    # Nucleus sampling threshold.
    mbr_top_p: float = 0.92

    # Derived number of stochastic candidates per model/sample.
    @property
    def num_sample_cands(self) -> int:
        return len(self.sample_temperatures) * self.num_sample_per_temp

    # MBR weights for selecting from the merged ensemble pool.
    # Maximum number of deduplicated candidates scored by MBR per sample.
    mbr_pool_cap: int = 48

    # Character n-gram similarity weight; useful for spelling/name variants.
    mbr_w_chrf: float = 0.55

    # BLEU similarity weight; captures word-level overlap and ordering.
    mbr_w_bleu: float = 0.25

    # Token-set overlap weight; rewards shared content words even if order differs.
    mbr_w_jaccard: float = 0.20

    # Length prior discourages extreme outliers.
    mbr_w_length: float = 0.10

    # Speed / memory switches.
    # Enable bf16 autocast when CUDA and the current GPU support it.
    use_mixed_precision: bool = True

    # Try BetterTransformer; fall back automatically if unavailable.
    use_better_transformer: bool = True

    # Group inputs by approximate length to reduce padding and wasted compute.
    use_bucket_batching: bool = True

    # Use fewer beams for short batches to reduce memory and latency.
    use_adaptive_beams: bool = True

    # Keep the stronger cleanup pipeline enabled for generated English text.
    aggressive_postprocessing: bool = True

    # Save an intermediate submission checkpoint after this many finalized rows.
    checkpoint_freq: int = 200

    def __post_init__(self):
        Path(self.output_dir).mkdir(exist_ok=True, parents=True)

        self.n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
        if self.n_gpus > 0:
            self.devices = [torch.device(f"cuda:{i}") for i in range(self.n_gpus)]
            self.device = self.devices[0]
        else:
            self.devices = [torch.device("cpu")]
            self.device = self.devices[0]

        if not torch.cuda.is_available():
            self.use_mixed_precision = False
            self.use_better_transformer = False

        self.use_bf16_amp = bool(
            self.use_mixed_precision
            and self.device.type == "cuda"
            and _supports_cuda_bf16(self.device)
        )

        assert self.num_beams >= self.num_beam_cands, "num_beams must be >= num_beam_cands"

        if self.use_diverse_beam:
            assert self.num_diverse_beams % self.num_beam_groups == 0
            assert self.num_diverse_beams >= self.num_diverse_cands

    def get_model_specs(self):
        # Fixed two-GPU assignment: cuda:0 -> A/C, cuda:1 -> B.
        if self.n_gpus >= 2:
            return [
                {
                    "label": "Model-A",
                    "path": self.model_a_path,
                    "max_new_tokens": self.model_a_max_new_tokens,
                    "device": torch.device("cuda:0"),
                },
                {
                    "label": "Model-B",
                    "path": self.model_b_path,
                    "max_new_tokens": self.model_b_max_new_tokens,
                    "device": torch.device("cuda:1"),
                },
                {
                    "label": "Model-C",
                    "path": self.model_c_path,
                    "max_new_tokens": self.model_c_max_new_tokens,
                    "device": torch.device("cuda:0"),
                },
            ]

        # Fall back to a single device when only one GPU or CPU is available.
        single_device = self.devices[0]
        return [
            {
                "label": "Model-A",
                "path": self.model_a_path,
                "max_new_tokens": self.model_a_max_new_tokens,
                "device": single_device,
            },
            {
                "label": "Model-B",
                "path": self.model_b_path,
                "max_new_tokens": self.model_b_max_new_tokens,
                "device": single_device,
            },
            {
                "label": "Model-C",
                "path": self.model_c_path,
                "max_new_tokens": self.model_c_max_new_tokens,
                "device": single_device,
            },
        ]


# =========================================================
# 3. Logging / Runtime summary
# =========================================================
def configure_inference_logging(output_dir: str) -> logging.Logger:
    Path(output_dir).mkdir(exist_ok=True, parents=True)
    for h in logging.root.handlers[:]:
        logging.root.removeHandler(h)

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
        handlers=[
            logging.StreamHandler(),
            logging.FileHandler(Path(output_dir) / "ensemble_mbr.log"),
        ],
    )
    return logging.getLogger("ensemble_mbr")


def print_runtime_summary(config: EnsembleInferenceConfig):
    print(f"PyTorch version : {torch.__version__}")
    print(f"CUDA available  : {torch.cuda.is_available()}")
    print(f"GPU count       : {config.n_gpus}")

    if torch.cuda.is_available():
        for i in range(config.n_gpus):
            device = torch.device(f"cuda:{i}")
            print(f"GPU {i} name    : {torch.cuda.get_device_name(i)}")
            total_memory_gb = torch.cuda.get_device_properties(i).total_memory / 1e9
            print(f"GPU {i} memory  : {total_memory_gb:.2f} GB")
            print(f"GPU {i} BF16    : {_supports_cuda_bf16(device)}")
    print(f"BF16 AMP enabled: {config.use_bf16_amp}")
    print()

    candidates_per_model = (
        config.num_beam_cands
        + (config.num_diverse_cands if config.use_diverse_beam else 0)
        + config.num_sample_cands
    )
    print(f"Candidates per model/sample: {candidates_per_model}")
    print(
        f"3-model merged candidate pool size: about "
        f"{candidates_per_model * len(config.get_model_specs())} before dedup"
    )
    print(f"Model-A max_new_tokens   : {config.model_a_max_new_tokens}")
    print(f"Model-B max_new_tokens   : {config.model_b_max_new_tokens}")
    print(f"Model-C max_new_tokens   : {config.model_c_max_new_tokens}")
    print()


# =========================================================
# 4. Preprocessing
# =========================================================
_V2 = re.compile(r"([aAeEiIuU])(?:2|₂)")
_V3 = re.compile(r"([aAeEiIuU])(?:3|₃)")
_ACUTE = str.maketrans({"a":"á","e":"é","i":"í","u":"ú","A":"Á","E":"É","I":"Í","U":"Ú"})
_GRAVE = str.maketrans({"a":"à","e":"è","i":"ì","u":"ù","A":"À","E":"È","I":"Ì","U":"Ù"})

def _restore_transliteration_diacritics(text: str) -> str:
    text = text.replace("sz", "š").replace("SZ", "Š")
    text = text.replace("s,", "ṣ").replace("S,", "Ṣ")
    text = text.replace("t,", "ṭ").replace("T,", "Ṭ")
    text = _V2.sub(lambda match: match.group(1).translate(_ACUTE), text)
    text = _V3.sub(lambda match: match.group(1).translate(_GRAVE), text)
    return text

_ALLOWED_FRACS = [
    (1/6, "0.16666"), (1/4, "0.25"), (1/3, "0.33333"),
    (1/2, "0.5"), (2/3, "0.66666"), (3/4, "0.75"), (5/6, "0.83333"),
]
_FRAC_TOL = 2e-3
_FLOAT_RE = re.compile(r"(?<![\w/])(\d+\.\d{4,})(?![\w/])")

def _canonicalize_decimal(value: float) -> str:
    integer_part = int(math.floor(value + 1e-12))
    fractional_part = value - integer_part
    nearest_fraction = min(_ALLOWED_FRACS, key=lambda item: abs(fractional_part - item[0]))
    if abs(fractional_part - nearest_fraction[0]) <= _FRAC_TOL:
        decimal_text = nearest_fraction[1]
        if integer_part == 0:
            return decimal_text
        return (
            f"{integer_part}{decimal_text[1:]}"
            if decimal_text.startswith("0.")
            else f"{integer_part}+{decimal_text}"
        )
    return f"{value:.5f}".rstrip("0").rstrip(".")

_WS_RE = re.compile(r"\s+")

_GAP_UNIFIED_RE = re.compile(
    r"<\s*big[\s_\-]*gap\s*>"
    r"|<\s*gap\s*>"
    r"|\bbig[\s_\-]*gap\b"
    r"|\bx(?:\s+x)+\b"
    r"|\.{3,}|…+|\[\.+\]"
    r"|\[\s*x\s*\]|\(\s*x\s*\)"
    r"|(?<!\w)x{2,}(?!\w)"
    r"|(?<!\w)x(?!\w)"
    r"|\(\s*large\s+break\s*\)"
    r"|\(\s*break\s*\)"
    r"|\(\s*\d+\s+broken\s+lines?\s*\)",
    re.I
)

def _normalize_gap_markers(text_series: pd.Series) -> pd.Series:
    return text_series.str.replace(_GAP_UNIFIED_RE, "<gap>", regex=True)

_CHAR_TRANS = str.maketrans({
    "ḫ":"h","Ḫ":"H","ʾ":"",
    "₀":"0","₁":"1","₂":"2","₃":"3","₄":"4",
    "₅":"5","₆":"6","₇":"7","₈":"8","₉":"9",
    "—":"-","–":"-",
})
_SUB_X = "ₓ"

_UNICODE_UPPER = r"A-ZŠṬṢḪ\u00C0-\u00D6\u00D8-\u00DE\u0160\u1E00-\u1EFF"
_UNICODE_LOWER = r"a-zšṭṣḫ\u00E0-\u00F6\u00F8-\u00FF\u0161\u1E01-\u1EFF"

_DET_UPPER_RE = re.compile(r"\(([" + _UNICODE_UPPER + r"0-9]{1,6})\)")
_DET_LOWER_RE = re.compile(r"\(([" + _UNICODE_LOWER + r"]{1,4})\)")

_PN_RE = re.compile(r"\bPN\b")
_KUBABBAR_RE = re.compile(r"KÙ\.B\.")
_EXACT_FRAC_RE = re.compile(r"0\.8333|0\.6666|0\.3333|0\.1666|0\.625|0\.75|0\.25|0\.5")
_EXACT_FRAC_MAP = {
    "0.8333": "⅚", "0.6666": "⅔", "0.3333": "⅓", "0.1666": "⅙",
    "0.625": "⅝", "0.75": "¾", "0.25": "¼", "0.5": "½",
}

def _replace_fraction(match: re.Match) -> str:
    return _EXACT_FRAC_MAP[match.group(0)]

# Normalize Akkadian notation before tokenization.
class AkkadianTextPreprocessor:
    def preprocess_batch(self, texts: List[str]) -> List[str]:
        text_series = pd.Series(texts).fillna("").astype(str)
        text_series = text_series.apply(_restore_transliteration_diacritics)
        text_series = text_series.str.replace(_DET_UPPER_RE, r"\1", regex=True)
        text_series = text_series.str.replace(_DET_LOWER_RE, r"{\1}", regex=True)
        text_series = _normalize_gap_markers(text_series)
        text_series = text_series.str.translate(_CHAR_TRANS)
        text_series = text_series.str.replace(_SUB_X, "", regex=False)
        text_series = text_series.str.replace(_KUBABBAR_RE, "KÙ.BABBAR", regex=True)
        text_series = text_series.str.replace(_EXACT_FRAC_RE, _replace_fraction, regex=True)
        text_series = text_series.str.replace(
            _FLOAT_RE,
            lambda match: _canonicalize_decimal(float(match.group(1))),
            regex=True,
        )
        text_series = text_series.str.replace(_WS_RE, " ", regex=True).str.strip()
        return text_series.tolist()


# =========================================================
# 5. Postprocessing
# =========================================================
# Remove editorial grammar notes from generated English.
_SOFT_GRAM_RE = re.compile(
    r"\(\s*(?:fem|plur|pl|sing|singular|plural|\?|\!)"
    r"(?:\.\s*(?:plur|plural|sing|singular))?"
    r"\.?\s*[^)]*\)", re.I
)
_BARE_GRAM_RE = re.compile(r"(?<!\w)(?:fem|sing|pl|plural)\.?(?!\w)\s*", re.I)
# Normalize uncertainty and quotation artifacts before scoring with BLEU/chrF++.
_UNCERTAIN_RE = re.compile(r"\(\?\)")
_CURLY_DQ_RE = re.compile("[\u201c\u201d]")
_CURLY_SQ_RE = re.compile("[\u2018\u2019]")
# Convert month Roman numerals to Arabic numerals.
_MONTH_RE = re.compile(r"\bMonth\s+(XII|XI|X|IX|VIII|VII|VI|V|IV|III|II|I)\b", re.I)
_ROMAN2INT = {"I":1,"II":2,"III":3,"IV":4,"V":5,"VI":6,"VII":7,"VIII":8,"IX":9,"X":10,"XI":11,"XII":12}
# Collapse local repetition artifacts from generation.
_REPEAT_WORD_RE = re.compile(r"\b(\w+)(?:\s+\1\b)+")
_REPEAT_PUNCT_RE = re.compile(r"([.,])\1+")
_PUNCT_SPACE_RE = re.compile(r"\s+([.,:])")
# Strip source-side editorial marks from English output.
_FORBIDDEN_TRANS = str.maketrans("", "", '——<>⌈⌋⌊[]+ʾ;')
# Expand short domain-specific commodity labels common in merchant-tablet data.
_COMMODITY_RE = re.compile(r'(?<=\s)-(gold|tax|textiles)\b')
_COMMODITY_REPL = {"gold": "pašallum gold", "tax": "šadduātum tax", "textiles": "kutānum textiles"}

def _expand_commodity_label(match: re.Match) -> str:
    return _COMMODITY_REPL[match.group(1)]

# Canonicalize frequent shekel/grain expressions.
_SHEKEL_REPLS = [
    (re.compile(r'5\s+11\s*/\s*12\s+shekels?', re.I), '6 shekels less 15 grains'),
    (re.compile(r'5\s*/\s*12\s+shekels?', re.I), '⅓ shekel 15 grains'),
    (re.compile(r'7\s*/\s*12\s+shekels?', re.I), '½ shekel 15 grains'),
    (re.compile(r'1\s*/\s*12\s*(?:\(shekel\)|\bshekel)?', re.I), '15 grains'),
]

# Clean slash alternatives, stray markup, repeated gaps, and damaged-sign placeholders.
_SLASH_ALT_RE = re.compile(r'(?<![0-9/])\s+/\s+(?![0-9])\S+')
_STRAY_MARKS_RE = re.compile(r'<<[^>]*>>|<(?!gap\b)[^>]*>')
_MULTI_GAP_RE = re.compile(r'(?:<gap>\s*){2,}')
_EXTRA_STRAY_RE = re.compile(r'(?<!\w)(?:\.\.+|xx+)(?!\w)')
_HACEK_TRANS = str.maketrans({"ḫ": "h", "Ḫ": "H"})

def _normalize_month_number(match: re.Match) -> str:
    return f"Month {_ROMAN2INT.get(match.group(1).upper(), match.group(1))}"

# Batch-clean generated English while preserving translation content.
class TranslationPostprocessor:
    def postprocess_batch(self, translations: List[str]) -> List[str]:
        translation_series = pd.Series(translations).fillna("").astype(str)
        translation_series = _normalize_gap_markers(translation_series)
        translation_series = translation_series.str.replace(_PN_RE, "<gap>", regex=True)
        translation_series = translation_series.str.replace(
            _COMMODITY_RE, _expand_commodity_label, regex=True
        )

        for pattern, replacement in _SHEKEL_REPLS:
            translation_series = translation_series.str.replace(pattern, replacement, regex=True)

        translation_series = translation_series.str.replace(_EXACT_FRAC_RE, _replace_fraction, regex=True)
        translation_series = translation_series.str.replace(
            _FLOAT_RE,
            lambda match: _canonicalize_decimal(float(match.group(1))),
            regex=True,
        )
        translation_series = translation_series.str.replace(_SOFT_GRAM_RE, " ", regex=True)
        translation_series = translation_series.str.replace(_BARE_GRAM_RE, " ", regex=True)
        translation_series = translation_series.str.replace(_UNCERTAIN_RE, "", regex=True)
        translation_series = translation_series.str.replace(_STRAY_MARKS_RE, "", regex=True)
        translation_series = translation_series.str.replace(_EXTRA_STRAY_RE, "", regex=True)
        translation_series = translation_series.str.replace(_SLASH_ALT_RE, "", regex=True)
        translation_series = translation_series.str.replace(_CURLY_DQ_RE, '"', regex=True)
        translation_series = translation_series.str.replace(_CURLY_SQ_RE, "'", regex=True)
        translation_series = translation_series.str.replace(_MONTH_RE, _normalize_month_number, regex=True)
        translation_series = translation_series.str.replace(_MULTI_GAP_RE, "<gap>", regex=True)

        translation_series = translation_series.str.replace("<gap>", "\x00GAP\x00", regex=False)
        translation_series = translation_series.str.translate(_FORBIDDEN_TRANS)
        translation_series = translation_series.str.replace("\x00GAP\x00", " <gap> ", regex=False)

        translation_series = translation_series.str.translate(_HACEK_TRANS)
        translation_series = translation_series.str.replace(_REPEAT_WORD_RE, r"\1", regex=True)

        for phrase_length in range(4, 1, -1):
            pattern = (
                r"\b((?:\w+\s+){"
                + str(phrase_length - 1)
                + r"}\w+)(?:\s+\1\b)+"
            )
            translation_series = translation_series.str.replace(pattern, r"\1", regex=True)

        translation_series = translation_series.str.replace(_PUNCT_SPACE_RE, r"\1", regex=True)
        translation_series = translation_series.str.replace(_REPEAT_PUNCT_RE, r"\1", regex=True)
        translation_series = translation_series.str.replace(_WS_RE, " ", regex=True).str.strip()
        return translation_series.tolist()


# =========================================================
# 6. Dataset / Bucket batching
# =========================================================
class AkkadianInferenceDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, preprocessor: AkkadianTextPreprocessor, logger: logging.Logger):
        self.sample_ids = dataframe["id"].tolist()
        normalized_transliterations = preprocessor.preprocess_batch(
            dataframe["transliteration"].tolist()
        )
        self.input_texts = [
            "translate Akkadian to English: " + text
            for text in normalized_transliterations
        ]
        logger.info(f"Dataset initialized: {len(self.sample_ids)} samples")

    def __len__(self):
        return len(self.sample_ids)

    def __getitem__(self, idx):
        return self.sample_ids[idx], self.input_texts[idx]

# Bucket inputs by approximate length to reduce padding during generation.
class LengthBucketBatchSampler(Sampler):
    def __init__(self, dataset, batch_size, num_buckets, logger, shuffle=False):
        self.batch_size = batch_size
        self.shuffle = shuffle
        lengths = [len(text.split()) for _, text in dataset]
        sorted_indices = sorted(range(len(lengths)), key=lambda i: lengths[i])
        bucket_size = max(1, len(sorted_indices) // max(1, num_buckets))
        self.buckets = [
            sorted_indices[
                i * bucket_size : None if i == num_buckets - 1 else (i + 1) * bucket_size
            ]
            for i in range(num_buckets)
        ]

        for i, bucket in enumerate(self.buckets):
            if bucket:
                bucket_lengths = [lengths[index] for index in bucket]
                logger.info(
                    f"  Bucket {i}: {len(bucket)} samples, length range "
                    f"[{min(bucket_lengths)}, {max(bucket_lengths)}]"
                )

    def __iter__(self):
        for bucket in self.buckets:
            bucket_indices = list(bucket)
            if self.shuffle:
                random.shuffle(bucket_indices)
            for i in range(0, len(bucket_indices), self.batch_size):
                yield bucket_indices[i : i + self.batch_size]

    def __len__(self):
        return sum(math.ceil(len(bucket) / self.batch_size) for bucket in self.buckets)


# =========================================================
# 7. Model wrapper
# =========================================================
class TranslationModelRunner:
    def __init__(
        self,
        model_path: str,
        config: EnsembleInferenceConfig,
        logger: logging.Logger,
        label: str,
        max_new_tokens: int,
        device: torch.device,
    ):
        self.config = config
        self.logger = logger
        self.label = label
        self.max_new_tokens = max_new_tokens
        self.device = device
        self.tokenizer = None
        self.model = None

        logger.info(f"[{label}] Loading model from {model_path} to {self.device} ...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(self.device).eval()

        if self.device.type == "cuda":
            try:
                torch.set_float32_matmul_precision("high")
            except Exception:
                pass

        parameter_count = sum(parameter.numel() for parameter in self.model.parameters())
        logger.info(f"[{label}] Parameters: {parameter_count:,}")
        logger.info(f"[{label}] device: {self.device}")
        logger.info(f"[{label}] max_new_tokens: {self.max_new_tokens}")

        if self.device.type == "cuda":
            allocated_memory_gb = torch.cuda.memory_allocated(self.device.index) / 1e9
            logger.info(f"[{label}] {self.device} allocated memory: {allocated_memory_gb:.2f} GB")

        if config.use_better_transformer and self.device.type == "cuda":
            try:
                from optimum.bettertransformer import BetterTransformer
                self.model = BetterTransformer.transform(self.model)
                logger.info(f"[{label}] BetterTransformer acceleration enabled")
            except Exception as e:
                logger.warning(f"[{label}] Skipped BetterTransformer: {e}")

    def collate_tokenized_batch(self, batch_samples):
        sample_ids = [sample[0] for sample in batch_samples]
        source_texts = [sample[1] for sample in batch_samples]
        tokenized_batch = self.tokenizer(
            source_texts,
            max_length=self.config.max_input_length,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )
        return sample_ids, tokenized_batch

    # Generate a candidate pool per sample, then merge pools across the 3-model ensemble for MBR selection.
    def generate_translation_candidates(self, input_ids, attention_mask, beam_size: int) -> List[List[str]]:
        config = self.config
        batch_size = input_ids.shape[0]
        autocast_context = _bfloat16_autocast_context(self.device, config.use_bf16_amp)

        beam_candidate_count = config.num_beam_cands
        diverse_candidate_count = config.num_diverse_cands if config.use_diverse_beam else 0
        samples_per_temperature = config.num_sample_per_temp

        with autocast_context:
            beam_outputs = self.model.generate( # Search beam_size paths and keep the top completed sequences.
                input_ids=input_ids,
                attention_mask=attention_mask,
                do_sample=False,
                num_beams=max(beam_size, beam_candidate_count),
                num_return_sequences=beam_candidate_count,
                max_new_tokens=self.max_new_tokens,
                length_penalty=config.length_penalty,
                early_stopping=config.early_stopping,
                repetition_penalty=config.repetition_penalty,
                use_cache=True,
            )
            beam_texts = self.tokenizer.batch_decode(beam_outputs, skip_special_tokens=True)

            diverse_texts = []
            generated_diverse_count = 0
            if config.use_diverse_beam:
                try:
                    diverse_outputs = self.model.generate(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        do_sample=False,
                        num_beams=config.num_diverse_beams,
                        num_beam_groups=config.num_beam_groups,
                        diversity_penalty=config.diversity_penalty,
                        num_return_sequences=config.num_diverse_cands,
                        max_new_tokens=self.max_new_tokens,
                        length_penalty=config.length_penalty,
                        early_stopping=config.early_stopping,
                        repetition_penalty=config.repetition_penalty,
                        use_cache=True,
                    )
                    diverse_texts = self.tokenizer.batch_decode(diverse_outputs, skip_special_tokens=True)
                    generated_diverse_count = config.num_diverse_cands
                except Exception as e:
                    self.logger.warning(f"[{self.label}] Diverse beam search failed ({e}); skipping")

            sampled_texts_by_temperature = []
            temperature_count = 0
            if config.use_sampling and config.sample_temperatures:
                temperature_count = len(config.sample_temperatures)
                for temperature in config.sample_temperatures:
                    try:
                        sampled_outputs = self.model.generate(
                            input_ids=input_ids,
                            attention_mask=attention_mask,
                            do_sample=True,
                            num_beams=1,
                            top_p=config.mbr_top_p,
                            temperature=temperature,
                            num_return_sequences=samples_per_temperature,
                            max_new_tokens=self.max_new_tokens,
                            repetition_penalty=config.repetition_penalty,
                            use_cache=True,
                        )
                        temperature_texts = self.tokenizer.batch_decode(sampled_outputs, skip_special_tokens=True)
                        sampled_texts_by_temperature.extend(temperature_texts)
                    except Exception as e:
                        self.logger.warning(f"[{self.label}] Sampling at temperature {temperature:.2f} failed ({e}); filling blanks")
                        sampled_texts_by_temperature.extend([""] * (batch_size * samples_per_temperature))

        candidate_pools = []
        for i in range(batch_size):
            candidate_pool = []
            candidate_pool.extend(beam_texts[i * beam_candidate_count : (i + 1) * beam_candidate_count])

            if diverse_texts and generated_diverse_count > 0: # only when use_diverse_beam
                candidate_pool.extend(diverse_texts[i * generated_diverse_count : (i + 1) * generated_diverse_count])

            '''
            temp=0.60:
              sample0_result0
              sample0_result1
              sample1_result0
              sample1_result1

            temp=0.80:
              sample0_result0
              sample0_result1
              sample1_result0
              sample1_result1

            temp=1.05:
              sample0_result0
              sample0_result1
              sample1_result0
              sample1_result1
            '''
            if sampled_texts_by_temperature and temperature_count > 0:
                for temperature_index in range(temperature_count):
                    sample_offset = (
                        temperature_index * batch_size * samples_per_temperature
                        + i * samples_per_temperature
                    )
                    candidate_pool.extend(
                        sampled_texts_by_temperature[
                            sample_offset : sample_offset + samples_per_temperature
                        ]
                    )

            candidate_pools.append(candidate_pool)

        if candidate_pools:
            self.logger.info(
                f"[{self.label}] Candidate pool: beam={beam_candidate_count} + diverse={generated_diverse_count} + "
                f"sample={temperature_count}x{samples_per_temperature}="
                f"{temperature_count*samples_per_temperature} = total {len(candidate_pools[0])}"
            )

        return candidate_pools

    def unload_model(self):
        label = self.label
        try:
            from optimum.bettertransformer import BetterTransformer
            self.model = BetterTransformer.reverse(self.model)
        except Exception:
            pass

        del self.model
        del self.tokenizer
        self.model = None
        self.tokenizer = None

        gc.collect()
        if torch.cuda.is_available() and self.device.type == "cuda":
            torch.cuda.empty_cache()
            torch.cuda.synchronize(self.device)
            total_memory = torch.cuda.get_device_properties(self.device.index).total_memory
            allocated_memory = torch.cuda.memory_allocated(self.device.index)
            free_memory_gb = (total_memory - allocated_memory) / 1e9
            self.logger.info(f"[{label}] Model unloaded. {self.device} free memory: {free_memory_gb:.2f} GB")


# =========================================================
# 8. MBR selector
# =========================================================
# MBR selects the candidate that best agrees with the rest of the pool.
# Agreement uses chrF++, BLEU, Jaccard, and a small length prior.
class MBRCandidateSelector:
    def __init__(self, pool_cap=48, w_chrf=0.55, w_bleu=0.25, w_jaccard=0.20, w_length=0.10):
        self._chrf_metric = sacrebleu.metrics.CHRF(word_order=2)
        self._bleu_metric = sacrebleu.metrics.BLEU(effective_order=True)
        self.pool_cap = pool_cap
        self.w_chrf = w_chrf
        self.w_bleu = w_bleu
        self.w_jaccard = w_jaccard
        self.w_length = w_length
        self._pw_total = max(w_chrf + w_bleu + w_jaccard, 1e-9)

    def _chrfpp_score(self, candidate_a: str, candidate_b: str) -> float:
        if not candidate_a or not candidate_b:
            return 0.0
        return float(self._chrf_metric.sentence_score(candidate_a, [candidate_b]).score)

    def _bleu_score(self, candidate_a: str, candidate_b: str) -> float:
        if not candidate_a or not candidate_b:
            return 0.0
        try:
            return float(self._bleu_metric.sentence_score(candidate_a, [candidate_b]).score)
        except Exception:
            return 0.0

    @staticmethod
    def _jaccard_score(candidate_a: str, candidate_b: str) -> float:
        tokens_a = set(candidate_a.lower().split())
        tokens_b = set(candidate_b.lower().split())
        if not tokens_a and not tokens_b:
            return 100.0
        if not tokens_a or not tokens_b:
            return 0.0
        return 100.0 * len(tokens_a & tokens_b) / len(tokens_a | tokens_b)

    def _candidate_agreement_score(self, candidate_a: str, candidate_b: str) -> float:
        weighted_score = (
            self.w_chrf * self._chrfpp_score(candidate_a, candidate_b)
            + self.w_bleu * self._bleu_score(candidate_a, candidate_b)
            + self.w_jaccard * self._jaccard_score(candidate_a, candidate_b)
        )
        return weighted_score / self._pw_total

    @staticmethod
    def _candidate_length_bonus(lengths: List[int], candidate_index: int) -> float:
        if len(lengths) == 0:
            return 100.0
        median = float(np.median(lengths))
        sigma = max(median * 0.4, 5.0)
        standardized_distance = (lengths[candidate_index] - median) / sigma
        return 100.0 * math.exp(-0.5 * standardized_distance * standardized_distance)

    @staticmethod
    def _deduplicate_candidates(candidates: List[str]) -> List[str]:
        seen, unique_candidates = set(), []
        for candidate in candidates:
            candidate = str(candidate).strip()
            if candidate and candidate not in seen:
                unique_candidates.append(candidate)
                seen.add(candidate)
        return unique_candidates

    def select_candidate(self, candidates: List[str]) -> str:
        unique_candidates = self._deduplicate_candidates(candidates)
        if self.pool_cap:
            unique_candidates = unique_candidates[:self.pool_cap]

        candidate_count = len(unique_candidates)
        if candidate_count == 0:
            return ""
        if candidate_count == 1:
            return unique_candidates[0]

        lengths = [len(candidate.split()) for candidate in unique_candidates]
        scores = []

        for i in range(candidate_count):
            agreement_score = sum(
                self._candidate_agreement_score(unique_candidates[i], unique_candidates[j])
                for j in range(candidate_count)
                if j != i
            ) / max(1, candidate_count - 1)
            length_bonus = self._candidate_length_bonus(lengths, i)
            total_score = agreement_score + self.w_length * length_bonus
            scores.append(total_score)

        return unique_candidates[int(np.argmax(scores))]


# =========================================================
# 9. Engine
# =========================================================
class EnsembleInferenceEngine:
    def __init__(self, config: EnsembleInferenceConfig, logger: logging.Logger):
        self.config = config
        self.logger = logger
        self.preprocessor = AkkadianTextPreprocessor()
        self.postprocessor = TranslationPostprocessor()
        self.mbr = MBRCandidateSelector(
            pool_cap=config.mbr_pool_cap,
            w_chrf=config.mbr_w_chrf,
            w_bleu=config.mbr_w_bleu,
            w_jaccard=config.mbr_w_jaccard,
            w_length=config.mbr_w_length,
        )

    def _select_beam_size(self, attention_mask: torch.Tensor) -> int:
        if not self.config.use_adaptive_beams:
            return self.config.num_beams
        median_input_length = float(attention_mask.sum(dim=1).float().median().item())
        short_batch_beam_size = max(
            self.config.num_beam_cands, self.config.num_beams // 2
        )
        return short_batch_beam_size if median_input_length < 100 else self.config.num_beams

    def _create_dataloader(
        self, dataset: AkkadianInferenceDataset, wrapper: TranslationModelRunner
    ) -> DataLoader:
        if self.config.use_bucket_batching:
            bucket_sampler = LengthBucketBatchSampler(
                dataset, self.config.batch_size, self.config.num_buckets, self.logger
            )
            return DataLoader(
                dataset,
                batch_sampler=bucket_sampler,
                num_workers=self.config.num_workers,
                collate_fn=wrapper.collate_tokenized_batch,
                pin_memory=(wrapper.device.type == "cuda"),
            )
        return DataLoader(
            dataset,
            batch_size=self.config.batch_size,
            shuffle=False,
            num_workers=self.config.num_workers,
            collate_fn=wrapper.collate_tokenized_batch,
            pin_memory=(wrapper.device.type == "cuda"),
        )

    def _generate_model_candidates(
        self, wrapper: TranslationModelRunner, dataset: AkkadianInferenceDataset
    ) -> Dict[str, List[str]]:
        """
        Run candidate generation for one already-loaded model wrapper.

        Returns a mapping from sample id to that model's candidate pool. Each
        pool is later merged with pools from the other ensemble models before
        postprocessing and MBR selection.
        """
        # Use this wrapper's tokenizer/collate function.
        dataloader = self._create_dataloader(dataset, wrapper)

        # Store per-sample candidate lists from this model.
        pools_by_id = {}

        # Disable autograd for lower inference memory use.
        with torch.inference_mode():
            for batch_ids, tokenized_batch in tqdm(
                dataloader, desc=f"  [{wrapper.label} @ {wrapper.device}]"
            ):
                # Move the batch to this wrapper's device.
                input_ids = tokenized_batch.input_ids.to(wrapper.device, non_blocking=True)
                attention_mask = tokenized_batch.attention_mask.to(
                    wrapper.device, non_blocking=True
                )

                # Short batches can use fewer beams.
                beam_size = self._select_beam_size(attention_mask)

                try:
                    # Generate configured beam/diverse/sampling candidates.
                    batch_candidate_pools = wrapper.generate_translation_candidates(
                        input_ids, attention_mask, beam_size
                    )
                    # Attach each candidate pool to its sample id.
                    for sample_id, candidate_pool in zip(batch_ids, batch_candidate_pools):
                        pools_by_id[str(sample_id)] = candidate_pool
                except RuntimeError as e:
                    if "out of memory" in str(e).lower():
                        self.logger.error(f"[{wrapper.label}] Out of memory (OOM); skipping this batch")
                        if wrapper.device.type == "cuda":
                            torch.cuda.empty_cache()
                        # Keep empty pools so merge/MBR can continue.
                        for sample_id in batch_ids:
                            pools_by_id.setdefault(str(sample_id), [])
                    else:
                        raise
                except Exception as e:
                    self.logger.error(f"[{wrapper.label}] Batch error: {e}")
                    # Represent non-OOM failures as empty pools too.
                    for sample_id in batch_ids:
                        pools_by_id.setdefault(str(sample_id), [])

                # Release cached CUDA blocks between batches.
                if wrapper.device.type == "cuda":
                    torch.cuda.empty_cache()

        return pools_by_id

    def _generate_device_candidates(
        self,
        model_wrappers: List[TranslationModelRunner],
        dataset: AkkadianInferenceDataset
    ) -> Dict[str, Dict[str, List[str]]]:
        """
        Run models sequentially on the same GPU and in parallel across GPUs.
        """
        model_candidate_pools = {}
        for wrapper in model_wrappers:
            self.logger.info(f"[{wrapper.device}] Starting inference for {wrapper.label}")
            model_candidate_pools[wrapper.label] = self._generate_model_candidates(wrapper, dataset)
            self.logger.info(f"[{wrapper.device}] Finished inference for {wrapper.label}")
        return model_candidate_pools

    def run_inference(self, test_dataframe: pd.DataFrame) -> pd.DataFrame:
        config, logger = self.config, self.logger
        dataset = AkkadianInferenceDataset(test_dataframe, self.preprocessor, logger)
        sample_ids = [str(sample_id) for sample_id in dataset.sample_ids]
        model_specs = config.get_model_specs()

        candidates_per_model = (
            config.num_beam_cands
            + (config.num_diverse_cands if config.use_diverse_beam else 0)
            + config.num_sample_cands
        )

        logger.info("=" * 70)
        logger.info("Ensemble x MBR | 3-model candidate pooling (A + B + C) | dual-GPU version")
        for model_spec in model_specs:
            logger.info(
                f"  {model_spec['label']}: path={model_spec['path']} | "
                f"max_new_tokens={model_spec['max_new_tokens']} | device={model_spec['device']}"
            )
        logger.info(f"  Candidates per model/sample: {candidates_per_model}")
        logger.info(
            f"  3-model merged candidate pool size before dedup: "
            f"{candidates_per_model * len(model_specs)}"
        )
        logger.info("=" * 70)

        # -------------------------------------------------
        # 1) Load all 3 models and keep them resident in memory.
        # -------------------------------------------------
        model_wrappers = []
        for model_spec in model_specs:
            model_wrappers.append(
                TranslationModelRunner(
                    model_path=model_spec["path"],
                    config=config,
                    logger=logger,
                    label=model_spec["label"],
                    max_new_tokens=model_spec["max_new_tokens"],
                    device=model_spec["device"],
                )
            )

        # Group wrappers by device.
        device_to_wrappers: Dict[str, List[TranslationModelRunner]] = {}
        for wrapper in model_wrappers:
            device_to_wrappers.setdefault(str(wrapper.device), []).append(wrapper)

        candidate_pools_by_model: Dict[str, Dict[str, List[str]]] = {}

        # -------------------------------------------------
        # 2) Run device groups in parallel; models on the same GPU run sequentially.
        # -------------------------------------------------
        logger.info("Stage 1/2 — Starting parallel dual-GPU inference")
        with ThreadPoolExecutor(max_workers=len(device_to_wrappers)) as executor:
            future_to_device = {
                executor.submit(
                    self._generate_device_candidates, group_wrappers, dataset
                ): device_name
                for device_name, group_wrappers in device_to_wrappers.items()
            }

            for future in as_completed(future_to_device):
                device_name = future_to_device[future]
                try:
                    device_results = future.result()
                    candidate_pools_by_model.update(device_results)
                    logger.info(f"[{device_name}] Finished all models on this device")
                except Exception as e:
                    logger.error(f"[{device_name}] Inference thread failed: {e}")
                    raise

        # -------------------------------------------------
        # 3) Unload models.
        # -------------------------------------------------
        logger.info("Stage 2/2 — Merge 3-model candidate pools and run MBR")
        for wrapper in model_wrappers:
            wrapper.unload_model()

        result_rows = []

        for sample_id in tqdm(sample_ids, desc="  Running MBR"):
            combined_candidates = []
            for model_spec in model_specs:
                combined_candidates.extend(
                    candidate_pools_by_model[model_spec["label"]].get(sample_id, [])
                )

            postprocessed_candidates = (
                self.postprocessor.postprocess_batch(combined_candidates)
                if combined_candidates
                else []
            )
            selected_translation = self.mbr.select_candidate(postprocessed_candidates)

            if not selected_translation or not selected_translation.strip():
                selected_translation = "The tablet is too damaged to translate."

            result_rows.append((sample_id, selected_translation))

            if len(result_rows) % config.checkpoint_freq == 0:
                checkpoint_path = (
                    Path(config.output_dir) / f"checkpoint_{len(result_rows)}.csv"
                )
                pd.DataFrame(result_rows, columns=["id", "translation"]).to_csv(
                    checkpoint_path, index=False
                )
                logger.info(
                    f"Checkpoint saved: {len(result_rows)} rows -> {checkpoint_path}"
                )

        result_dataframe = pd.DataFrame(result_rows, columns=["id", "translation"])
        self._log_result_summary(result_dataframe)
        return result_dataframe

    def _log_result_summary(self, result_dataframe: pd.DataFrame):
        logger = self.logger
        logger.info("=" * 60)
        empty_count = result_dataframe["translation"].str.strip().eq("").sum()
        translation_lengths = result_dataframe["translation"].str.len()
        logger.info(
            f"Empty outputs: {empty_count} "
            f"({100*empty_count/max(1,len(result_dataframe)):.2f}%)"
        )
        logger.info(
            f"Length stats: mean {translation_lengths.mean():.1f} | "
            f"median {translation_lengths.median():.1f} | max {translation_lengths.max()}"
        )
        for idx in [0, len(result_dataframe) // 2, len(result_dataframe) - 1]:
            row = result_dataframe.iloc[idx]
            logger.info(f"  Preview ID {row['id']}: {str(row['translation'])[:80]}...")
        logger.info("=" * 60)


# =========================================================
# 10. Main
# =========================================================
def main():
    config = EnsembleInferenceConfig()
    logger = configure_inference_logging(config.output_dir)

    print_runtime_summary(config)

    logger.info(f"Loading test data: {config.test_data_path}")
    test_dataframe = pd.read_csv(config.test_data_path, encoding="utf-8")
    logger.info(f"Samples to translate: {len(test_dataframe)}")

    inference_engine = EnsembleInferenceEngine(config, logger)
    result_dataframe = inference_engine.run_inference(test_dataframe)

    submission_path = Path(config.output_dir) / "submission.csv"
    result_dataframe.to_csv(submission_path, index=False)
    logger.info(
        f"Submission saved -> {submission_path} ({len(result_dataframe)} rows)"
    )

    config_snapshot = {
        "model_a": config.model_a_path,
        "model_b": config.model_b_path,
        "model_c": config.model_c_path,
        "model_a_max_new_tokens": config.model_a_max_new_tokens,
        "model_b_max_new_tokens": config.model_b_max_new_tokens,
        "model_c_max_new_tokens": config.model_c_max_new_tokens,
        "model_count": len(config.get_model_specs()),
        "num_beam_cands": config.num_beam_cands,
        "num_beams": config.num_beams,
        "length_penalty": config.length_penalty,
        "repetition_penalty": config.repetition_penalty,
        "use_diverse_beam": config.use_diverse_beam,
        "num_diverse_cands": config.num_diverse_cands,
        "num_diverse_beams": config.num_diverse_beams,
        "num_beam_groups": config.num_beam_groups,
        "diversity_penalty": config.diversity_penalty,
        "use_sampling": config.use_sampling,
        "sample_temperatures": config.sample_temperatures,
        "num_sample_per_temp": config.num_sample_per_temp,
        "num_sample_cands": config.num_sample_cands,
        "mbr_top_p": config.mbr_top_p,
        "mbr_w_chrf": config.mbr_w_chrf,
        "mbr_w_bleu": config.mbr_w_bleu,
        "mbr_w_jaccard": config.mbr_w_jaccard,
        "mbr_w_length": config.mbr_w_length,
        "mbr_pool_cap": config.mbr_pool_cap,
        "use_bf16_amp": config.use_bf16_amp,
        "batch_size": config.batch_size,
        "n_gpus": config.n_gpus,
    }

    config_json_path = Path(config.output_dir) / "ensemble_mbr_config.json"
    with open(config_json_path, "w") as config_file:
        json.dump(config_snapshot, config_file, indent=2)

    print("Submission path:", submission_path)
    print("Run config saved to:", config_json_path)


if __name__ == "__main__":
    main()
